# TCN Multi-Output — GWO-Optimized Final Evaluation 2 (H=10)

**Author:** Ibrahim Hanafy  
**Date:** August 2026  

**Best hyperparameters from GWO Search 2 (harder pilot set with patient 101):**

| Parameter | Value | Source |
|-----------|-------|--------|
| `n_filters` | 128 | GWO Search 2 (Eval 29) |
| `dropout` | 0.056 | GWO Search 2 |
| `lr` | 0.002259 | GWO Search 2 |
| `n_blocks` | 3 | Fixed (RF=29 ≫ lookback=10) |
| `kernel_size` | 3 | Fixed |
| `batch_size` | 128 | Fixed |

**Pilot set:** `['101', '103', '108', '119', '124']` — includes **patient 101** (harder morphology)  
**Pilot RMSE:** 0.030622  

**Evaluation:**
1. **Part A** — Standard 21 patients (excl. 111 & 118) × H=10 → for direct comparison with Search 1 / baseline
2. **Part B** — All MIT-BIH patients (auto-discovered) × H=10 → full dataset evaluation

---

## 1 — Imports & Setup

In [ ]:
# ── Install & Imports ──────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'wfdb', 'tqdm', 'seaborn', '--quiet'])

import os, time, gc, warnings, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import wfdb
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.stats import wilcoxon
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)

SEED = 42
np.random.seed(SEED)
print('All imports OK.')

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import (
    Input, Conv1D, Dense, Dropout, Add, Activation, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU(s): {[g.name for g in gpus]}')
else:
    print('No GPU — training will be slow.')

## 2 — Configuration

In [ ]:
# ── Dataset ─────────────────────────────────────────────────────────────────
DATA_DIR = r"/kaggle/input/datasets/rracer17/mit-bih-mitdb/mit-bih-arrhythmia-database-1.0.0"

# ── Paper constants ────────────────────────────────────────────────────────
FS           = 360
TOTAL_STEPS  = 100_000
TRAIN_STEPS  = 40_000
VAL_STEPS    = 10_000
TEST_STEPS   = 50_000
LOOKBACK     = 10

# ── Standard 21 patients (excluding 111 & 118) ────────────────────────────
PATIENTS_21 = [
    '100', '101', '102', '103', '104', '105',
    '106', '107', '108', '109', '112', '113',
    '114', '115', '116', '117', '119',
    '121', '122', '123', '124'
]
assert len(PATIENTS_21) == 21

# ── Auto-discover ALL patients in MIT-BIH dataset ──────────────────────────
ALL_PATIENTS = sorted(set(
    os.path.splitext(os.path.basename(f))[0]
    for f in glob.glob(os.path.join(DATA_DIR, '*.hea'))
))
print(f'Standard patients: {len(PATIENTS_21)}')
print(f'Discovered patients: {len(ALL_PATIENTS)} → {ALL_PATIENTS}')

# ── Horizon ────────────────────────────────────────────────────────────────
HORIZONS = [10]

# ┌──────────────────────────────────────────────────────────────────────────┐
# │  GWO SEARCH 2 — BEST HYPERPARAMETERS                                  │
# │  Source: GWO Search 2 (Eval 29), pilot RMSE = 0.030622                │
# │  Pilot: ['101','103','108','119','124'] (harder set with patient 101)  │
# │  Search: 8 wolves × 4 epochs, n_blocks=3 fixed                        │
# └──────────────────────────────────────────────────────────────────────────┘
NUM_FILTERS   = 128
DROPOUT_RATE  = 0.056
LEARNING_RATE = 0.002259
NUM_BLOCKS    = 3       # RF = 1 + 2(3-1)(2^3-1) = 29 >> lookback=10
KERNEL_SIZE   = 3
BATCH_SIZE    = 128

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS   = 200
PATIENCE = 50

# ── Output ─────────────────────────────────────────────────────────────────
CSV_21   = 'tcn_mo_gwo2_optimized_21patients_results.csv'
CSV_ALL  = 'tcn_mo_gwo2_optimized_all_patients_results.csv'
PLOT_DIR = 'plots_gwo2_optimized'
os.makedirs(PLOT_DIR, exist_ok=True)

print(f'\nConfig    : GWO Search 2 Optimized')
print(f'Horizons  : {HORIZONS}')
print(f'TCN       : blocks={NUM_BLOCKS}, filters={NUM_FILTERS}, kernel={KERNEL_SIZE}')
print(f'Dropout   : {DROPOUT_RATE}')
print(f'Training  : epochs={EPOCHS}, patience={PATIENCE}, batch={BATCH_SIZE}, lr={LEARNING_RATE}')

## 3 — Data Pipeline

In [ ]:
def load_ecg_signal(record_id, data_dir, n_steps=100_000):
    """Load MLII lead from MIT-BIH, truncate/pad to n_steps."""
    path = os.path.join(data_dir, record_id)
    rec  = wfdb.rdrecord(path)
    sig_names_upper = [s.upper() for s in rec.sig_name]
    ch = sig_names_upper.index('MLII') if 'MLII' in sig_names_upper else 0
    signal = rec.p_signal[:, ch].astype(np.float32)
    if len(signal) < n_steps:
        pad = np.full(n_steps - len(signal), signal[-1], dtype=np.float32)
        signal = np.concatenate([signal, pad])
    return signal[:n_steps]


def preprocess_patient(signal, train_steps=40_000, val_steps=10_000):
    """Split → train/val/test, MinMax-normalise (fit on train only)."""
    train_end = train_steps
    val_end   = train_steps + val_steps
    train_raw = signal[:train_end]
    val_raw   = signal[train_end:val_end]
    test_raw  = signal[val_end:]
    scaler     = MinMaxScaler(feature_range=(0, 1))
    train_norm = scaler.fit_transform(train_raw.reshape(-1, 1)).flatten()
    val_norm   = scaler.transform(val_raw.reshape(-1, 1)).flatten()
    test_norm  = scaler.transform(test_raw.reshape(-1, 1)).flatten()
    return train_norm, val_norm, test_norm, scaler


def make_multistep_sequences(signal, lookback, horizon):
    """X = lookback window, y = next H steps (Multi-Output)."""
    X, y = [], []
    for i in range(len(signal) - lookback - horizon + 1):
        X.append(signal[i : i + lookback])
        y.append(signal[i + lookback : i + lookback + horizon])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


print('Preprocessing functions defined.')

In [ ]:
# ── Load ALL patients (superset — includes the 21 standard) ────────────────
patient_signals = {}

print(f'Loading {len(ALL_PATIENTS)} patients...\n')
for rid in tqdm(ALL_PATIENTS, desc='Patients'):
    signal = load_ecg_signal(rid, DATA_DIR, n_steps=TOTAL_STEPS)
    tr, vl, te, sc = preprocess_patient(signal, TRAIN_STEPS, VAL_STEPS)
    patient_signals[rid] = {'train': tr, 'val': vl, 'test': te, 'scaler': sc}
    tqdm.write(f'  Patient {rid:>3s} | train={len(tr):,}  val={len(vl):,}  test={len(te):,}')

print(f'\nAll {len(patient_signals)} patients loaded.')
print(f'Standard 21 present: {all(p in patient_signals for p in PATIENTS_21)}')

## 4 — Model Architecture

In [ ]:
def residual_block(x, filters, kernel_size, dilation_rate, dropout_rate):
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(x)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    out = Conv1D(filters, kernel_size, dilation_rate=dilation_rate, padding='causal')(out)
    out = BatchNormalization()(out)
    out = Activation('relu')(out)
    out = Dropout(dropout_rate)(out)
    if x.shape[-1] != filters:
        x = Conv1D(filters, 1)(x)
    return Add()([x, out])


def build_tcn(lookback, output_size):
    inp = Input(shape=(lookback, 1))
    x = inp
    for i in range(NUM_BLOCKS):
        x = residual_block(x, NUM_FILTERS, KERNEL_SIZE, 2 ** i, DROPOUT_RATE)
    x = x[:, -1, :]  # last time-step
    x = Dense(NUM_FILTERS, activation='relu')(x)
    out = Dense(output_size)(x)
    model = Model(inp, out, name=f'TCN_out{output_size}')
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss='mse')
    return model


def train_tcn(model, X_train, y_train, X_val, y_val):
    X_tr = X_train.reshape(-1, X_train.shape[1], 1)
    X_vl = X_val.reshape(-1, X_val.shape[1], 1)
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=max(PATIENCE // 2, 2), min_lr=1e-6),
    ]
    history = model.fit(
        X_tr, y_train, validation_data=(X_vl, y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=0
    )
    return history


def compute_metrics(y_true, y_pred):
    yt, yp = y_true.flatten(), y_pred.flatten()
    return {
        'RMSE': float(np.sqrt(mean_squared_error(yt, yp))),
        'MAE':  float(mean_absolute_error(yt, yp)),
        'R2':   float(r2_score(yt, yp)),
    }


def compute_per_step_metrics(y_true, y_pred):
    rows = []
    for h in range(y_true.shape[1]):
        m = compute_metrics(y_true[:, h], y_pred[:, h])
        m['Step'] = h + 1
        rows.append(m)
    return pd.DataFrame(rows)


# Show model summary
_sample = build_tcn(LOOKBACK, 10)
_sample.summary()
print(f'\nTotal parameters: {_sample.count_params():,}')
del _sample
tf.keras.backend.clear_session()
gc.collect()

---

## 5A — Train & Evaluate: Standard 21 Patients (for comparison)

Same 21 patients as baseline and Search 1 — enables direct, fair comparison.

In [ ]:
def run_evaluation(patient_list, label):
    """Train and evaluate TCN on given patient list. Returns DataFrame of results."""
    results = []
    per_step_results = {}
    saved_preds = {}
    training_histories = {}

    total_runs = len(patient_list) * len(HORIZONS)
    print(f'\n{"=" * 80}')
    print(f'Running [{label}]: {len(patient_list)} patients × {len(HORIZONS)} horizons = {total_runs} models')
    print(f'TCN: {NUM_BLOCKS} blocks, {NUM_FILTERS} filters, dropout={DROPOUT_RATE}, lr={LEARNING_RATE}')
    print(f'{"=" * 80}')

    for rid in tqdm(patient_list, desc=label):
        tr = patient_signals[rid]['train']
        vl = patient_signals[rid]['val']
        te = patient_signals[rid]['test']

        for horizon in HORIZONS:
            X_tr, y_tr = make_multistep_sequences(tr, LOOKBACK, horizon)
            X_vl, y_vl = make_multistep_sequences(vl, LOOKBACK, horizon)
            X_te, y_te = make_multistep_sequences(te, LOOKBACK, horizon)

            t0 = time.time()
            model = build_tcn(LOOKBACK, horizon)
            history = train_tcn(model, X_tr, y_tr, X_vl, y_vl)
            preds = model.predict(
                X_te.reshape(-1, X_te.shape[1], 1),
                batch_size=2048, verbose=0
            )
            elapsed = time.time() - t0

            m = compute_metrics(y_te, preds)
            m['Time_s'] = round(elapsed, 2)
            results.append({'Patient': rid, 'Horizon': horizon, **m})

            if horizon > 1:
                per_step_results[(rid, horizon)] = compute_per_step_metrics(y_te, preds)

            saved_preds[(rid, horizon)] = (y_te.copy(), preds.copy())
            training_histories[(rid, horizon)] = {
                'loss': history.history['loss'],
                'val_loss': history.history['val_loss'],
            }

            del model
            gc.collect()

            tqdm.write(
                f'  Patient {rid} H={horizon:2d} | '
                f'R²={m["R2"]:.4f}  RMSE={m["RMSE"]:.4f}  MAE={m["MAE"]:.4f}  '
                f'({elapsed:.1f}s)'
            )

        tf.keras.backend.clear_session()
        gc.collect()

    df = pd.DataFrame(results)
    print(f'\n[{label}] Complete — {len(results)} experiments.')
    print(f'Mean RMSE: {df.RMSE.mean():.6f}  |  Mean R²: {df.R2.mean():.4f}')
    return df, per_step_results, saved_preds, training_histories

In [ ]:
# ── Part A: Standard 21 patients ──────────────────────────────────────────
df_21, perstep_21, preds_21, histories_21 = run_evaluation(PATIENTS_21, '21 Standard Patients')

df_21.to_csv(CSV_21, index=False)
print(f'\nResults saved → {CSV_21}')

## 6A — Comparison: 21-Patient Results (Search 1 vs Search 2 vs Baseline)

In [ ]:
# ┌──────────────────────────────────────────────────────────────────────────┐
# │  SEARCH 1 GWO RESULTS (from Final_Optimized.ipynb)                     │
# │  Config: 256 filters, 4 blocks, dropout=0.109, lr=0.002066            │
# │  Pilot: ['100','103','108','119','124'] (easier set)                   │
# └──────────────────────────────────────────────────────────────────────────┘
SEARCH1_RESULTS = {
    # Patient: (RMSE, MAE, R2, Time_s) — paste from Final_Optimized output
    # TODO: Fill in from Final_Optimized.ipynb results if CSV not available
}

# Try loading from CSV first
search1_csv = None
for path in ['tcn_mo_gwo_optimized_results.csv',
             '../tcn_mo_gwo_optimized_results.csv',
             'Figures/results/tcn_mo_gwo_optimized_results.csv',
             '../Figures/results/tcn_mo_gwo_optimized_results.csv']:
    if os.path.exists(path):
        search1_csv = path
        break

baseline_csv = None
for path in ['tcn_mo_ours_config_results.csv',
             '../tcn_mo_ours_config_results.csv',
             'Figures/results/tcn_mo_ours_config_results.csv',
             '../Figures/results/tcn_mo_ours_config_results.csv']:
    if os.path.exists(path):
        baseline_csv = path
        break

print('=== Configuration Comparison ===')
print(f'{"":>12s}  {"Baseline (Ours)":>16s}  {"Search 1 (GWO)":>16s}  {"Search 2 (GWO)":>16s}')
print(f'{"Blocks":>12s}  {"3":>16s}  {"4":>16s}  {"3":>16s}')
print(f'{"Filters":>12s}  {"64":>16s}  {"256":>16s}  {"128":>16s}')
print(f'{"Dropout":>12s}  {"0.1":>16s}  {"0.109":>16s}  {"0.056":>16s}')
print(f'{"LR":>12s}  {"0.001":>16s}  {"0.002066":>16s}  {"0.002259":>16s}')
print(f'{"Batch":>12s}  {"256":>16s}  {"128":>16s}  {"128":>16s}')
print(f'{"RF":>12s}  {"29":>16s}  {"61":>16s}  {"29":>16s}')
print()

In [ ]:
# ── Load comparison data ───────────────────────────────────────────────────
comparison_dfs = {'Search 2 (this)': df_21}

if search1_csv:
    df_s1 = pd.read_csv(search1_csv)
    if 'Horizon' in df_s1.columns:
        df_s1 = df_s1[df_s1.Horizon == 10]
    comparison_dfs['Search 1'] = df_s1
    print(f'Search 1 results loaded: {search1_csv} ({len(df_s1)} rows)')
else:
    print('⚠ Search 1 CSV not found — comparison will be partial.')

if baseline_csv:
    df_bl = pd.read_csv(baseline_csv)
    if 'Horizon' in df_bl.columns:
        df_bl = df_bl[df_bl.Horizon == 10]
    comparison_dfs['Baseline (Ours)'] = df_bl
    print(f'Baseline results loaded: {baseline_csv} ({len(df_bl)} rows)')
else:
    print('⚠ Baseline CSV not found — comparison will be partial.')

# ── Print summary ─────────────────────────────────────────────────────────
print(f'\n{"=" * 70}')
print(f'Mean Metrics Comparison (H=10, 21 patients)')
print(f'{"=" * 70}')
print(f'{"Config":>20s}  {"RMSE":>10s}  {"MAE":>10s}  {"R²":>10s}')
print(f'{"-" * 55}')
for name, df in comparison_dfs.items():
    print(f'{name:>20s}  {df.RMSE.mean():10.6f}  {df.MAE.mean():10.6f}  {df.R2.mean():10.4f}')
print(f'{"=" * 70}')

In [ ]:
# ── Per-patient comparison table ───────────────────────────────────────────
merge_cols = ['Patient']
df_compare = df_21[['Patient', 'RMSE', 'MAE', 'R2']].rename(
    columns={'RMSE': 'RMSE_S2', 'MAE': 'MAE_S2', 'R2': 'R2_S2'}
)

if search1_csv:
    df_compare = df_compare.merge(
        df_s1[['Patient', 'RMSE', 'R2']].rename(
            columns={'RMSE': 'RMSE_S1', 'R2': 'R2_S1'}
        ),
        on='Patient', how='left'
    )
    df_compare['ΔRMSE_S2vsS1'] = df_compare['RMSE_S2'] - df_compare['RMSE_S1']
    df_compare['ΔR2_S2vsS1']   = df_compare['R2_S2'] - df_compare['R2_S1']

if baseline_csv:
    df_compare = df_compare.merge(
        df_bl[['Patient', 'RMSE', 'R2']].rename(
            columns={'RMSE': 'RMSE_BL', 'R2': 'R2_BL'}
        ),
        on='Patient', how='left'
    )
    df_compare['ΔRMSE_S2vsBL'] = df_compare['RMSE_S2'] - df_compare['RMSE_BL']
    df_compare['ΔR2_S2vsBL']   = df_compare['R2_S2'] - df_compare['R2_BL']

display(df_compare.round(6))

In [ ]:
# ── Statistical significance (Wilcoxon signed-rank) ───────────────────────
s2_rmse = df_21.sort_values('Patient').RMSE.values

if search1_csv:
    s1_rmse = df_s1.sort_values('Patient').RMSE.values
    if len(s1_rmse) == len(s2_rmse):
        stat, p = wilcoxon(s1_rmse, s2_rmse)
        sig = '✓ significant' if p < 0.05 else '✗ not significant'
        better = 'Search 2' if s2_rmse.mean() < s1_rmse.mean() else 'Search 1'
        print(f'Search 1 vs Search 2 (Wilcoxon): p={p:.4f} → {sig} | Winner: {better}')
    else:
        print(f'⚠ Patient count mismatch: S1={len(s1_rmse)}, S2={len(s2_rmse)}')

if baseline_csv:
    bl_rmse = df_bl.sort_values('Patient').RMSE.values
    if len(bl_rmse) == len(s2_rmse):
        stat, p = wilcoxon(bl_rmse, s2_rmse)
        sig = '✓ significant' if p < 0.05 else '✗ not significant'
        better = 'Search 2' if s2_rmse.mean() < bl_rmse.mean() else 'Baseline'
        print(f'Baseline vs Search 2 (Wilcoxon): p={p:.4f} → {sig} | Winner: {better}')
    else:
        print(f'⚠ Patient count mismatch: BL={len(bl_rmse)}, S2={len(s2_rmse)}')

## 7A — Visualizations (21 Patients)

In [ ]:
# ── 7A.1 Per-Patient RMSE Bar Chart ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
df_sorted = df_21.sort_values('RMSE')
bars = ax.bar(df_sorted.Patient, df_sorted.RMSE, color='steelblue', edgecolor='white')
ax.set_xlabel('Patient')
ax.set_ylabel('RMSE')
ax.set_title('GWO Search 2 — Per-Patient RMSE (H=10, 21 Patients)', fontweight='bold')
ax.axhline(df_21.RMSE.mean(), color='red', ls='--', lw=1.5, label=f'Mean={df_21.RMSE.mean():.4f}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/rmse_bar_21.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7A.2 Per-Patient R² Bar Chart ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
df_sorted = df_21.sort_values('R2', ascending=False)
colors = ['forestgreen' if r > 0.99 else 'steelblue' if r > 0.95 else 'orange' if r > 0.9 else 'red'
          for r in df_sorted.R2]
ax.bar(df_sorted.Patient, df_sorted.R2, color=colors, edgecolor='white')
ax.set_xlabel('Patient')
ax.set_ylabel('R²')
ax.set_title('GWO Search 2 — Per-Patient R² (H=10, 21 Patients)', fontweight='bold')
ax.axhline(df_21.R2.mean(), color='red', ls='--', lw=1.5, label=f'Mean={df_21.R2.mean():.4f}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/r2_bar_21.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7A.3 Comparison Bar Chart (if comparison data available) ───────────────
if search1_csv or baseline_csv:
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    # RMSE comparison
    ax = axes[0]
    x = np.arange(len(PATIENTS_21))
    width = 0.25
    offset = 0

    if baseline_csv:
        bl_vals = df_bl.sort_values('Patient').RMSE.values
        ax.bar(x + offset * width, bl_vals, width, label='Baseline (Ours)', alpha=0.8)
        offset += 1
    if search1_csv:
        s1_vals = df_s1.sort_values('Patient').RMSE.values
        ax.bar(x + offset * width, s1_vals, width, label='Search 1 (GWO)', alpha=0.8)
        offset += 1

    s2_vals = df_21.sort_values('Patient').RMSE.values
    ax.bar(x + offset * width, s2_vals, width, label='Search 2 (GWO)', alpha=0.8)

    ax.set_xticks(x + width)
    ax.set_xticklabels(sorted(PATIENTS_21), rotation=45, ha='right')
    ax.set_ylabel('RMSE')
    ax.set_title('RMSE Comparison (H=10)', fontweight='bold')
    ax.legend()

    # R² comparison
    ax = axes[1]
    offset = 0

    if baseline_csv:
        bl_vals = df_bl.sort_values('Patient').R2.values
        ax.bar(x + offset * width, bl_vals, width, label='Baseline (Ours)', alpha=0.8)
        offset += 1
    if search1_csv:
        s1_vals = df_s1.sort_values('Patient').R2.values
        ax.bar(x + offset * width, s1_vals, width, label='Search 1 (GWO)', alpha=0.8)
        offset += 1

    s2_vals = df_21.sort_values('Patient').R2.values
    ax.bar(x + offset * width, s2_vals, width, label='Search 2 (GWO)', alpha=0.8)

    ax.set_xticks(x + width)
    ax.set_xticklabels(sorted(PATIENTS_21), rotation=45, ha='right')
    ax.set_ylabel('R²')
    ax.set_title('R² Comparison (H=10)', fontweight='bold')
    ax.legend()

    plt.suptitle('Search 2 (128f, 3b, pilot w/ 101) vs Previous Configs',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/comparison_bar_21.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No comparison CSVs found — skipping comparison chart.')

In [ ]:
# ── 7A.4 Training Curves (all 21 patients) ────────────────────────────────
fig, axes = plt.subplots(3, 7, figsize=(28, 10), sharex=True)
axes_flat = axes.flatten()

for idx, rid in enumerate(PATIENTS_21):
    ax = axes_flat[idx]
    h_key = (rid, 10)
    if h_key in histories_21:
        ax.plot(histories_21[h_key]['loss'], label='train', lw=0.8)
        ax.plot(histories_21[h_key]['val_loss'], label='val', lw=0.8)
    ax.set_title(f'P{rid}', fontsize=9)
    ax.tick_params(labelsize=7)
    if idx == 0:
        ax.legend(fontsize=7)

fig.suptitle('Training Curves — GWO Search 2 (21 Patients, H=10)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/training_curves_21.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7A.5 Per-Step RMSE Degradation ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

all_step_rmse = []
for rid in PATIENTS_21:
    key = (rid, 10)
    if key in perstep_21:
        df_ps = perstep_21[key]
        ax.plot(df_ps.Step, df_ps.RMSE, alpha=0.3, color='steelblue', lw=0.8)
        all_step_rmse.append(df_ps.RMSE.values)

if all_step_rmse:
    mean_rmse = np.mean(all_step_rmse, axis=0)
    std_rmse = np.std(all_step_rmse, axis=0)
    steps = np.arange(1, 11)
    ax.plot(steps, mean_rmse, color='red', lw=2.5, label='Mean ± 1σ')
    ax.fill_between(steps, mean_rmse - std_rmse, mean_rmse + std_rmse,
                    color='red', alpha=0.15)

ax.set_xlabel('Forecast Step')
ax.set_ylabel('RMSE')
ax.set_title('Per-Step RMSE Degradation — GWO Search 2 (21 Patients, H=10)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/perstep_rmse_21.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7A.6 Prediction Samples (4 representative patients) ───────────────────
sample_patients = ['100', '103', '108', '117']
sample_patients = [p for p in sample_patients if (p, 10) in preds_21]

if sample_patients:
    fig, axes = plt.subplots(len(sample_patients), 1, figsize=(16, 3.5 * len(sample_patients)))
    if len(sample_patients) == 1:
        axes = [axes]

    for ax, rid in zip(axes, sample_patients):
        y_true, y_pred = preds_21[(rid, 10)]
        n_show = min(500, len(y_true))
        ax.plot(y_true[:n_show, 0], label='Actual', lw=0.8, alpha=0.8)
        ax.plot(y_pred[:n_show, 0], label='Predicted', lw=0.8, alpha=0.8)
        rmse = np.sqrt(mean_squared_error(y_true.flatten(), y_pred.flatten()))
        ax.set_title(f'Patient {rid} — Step 1 (RMSE={rmse:.4f})', fontweight='bold')
        ax.legend()

    plt.suptitle('Actual vs Predicted — GWO Search 2', fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{PLOT_DIR}/predictions_21.png', dpi=150, bbox_inches='tight')
    plt.show()

---

## 5B — Train & Evaluate: ALL Patients (Full MIT-BIH Dataset)

Auto-discovered patients — includes all records in the dataset beyond the standard 21.

In [ ]:
# ── Find patients not yet evaluated ────────────────────────────────────────
extra_patients = [p for p in ALL_PATIENTS if p not in PATIENTS_21]
print(f'Standard 21: already evaluated above.')
print(f'Extra patients to evaluate: {len(extra_patients)} → {extra_patients}')

if extra_patients:
    df_extra, perstep_extra, preds_extra, histories_extra = run_evaluation(
        extra_patients, 'Extra Patients'
    )
    # Combine
    df_all = pd.concat([df_21, df_extra], ignore_index=True)
else:
    print('No extra patients — all patients are in the standard 21.')
    df_extra = pd.DataFrame()
    df_all = df_21.copy()

df_all.to_csv(CSV_ALL, index=False)
print(f'\nAll-patient results saved → {CSV_ALL}')
print(f'Total patients evaluated: {df_all.Patient.nunique()}')
print(f'Mean RMSE (all): {df_all.RMSE.mean():.6f}')
print(f'Mean R²   (all): {df_all.R2.mean():.4f}')

## 7B — Visualizations (All Patients)

In [ ]:
# ── 7B.1 Per-Patient RMSE Bar — ALL patients ──────────────────────────────
fig, ax = plt.subplots(figsize=(max(16, len(df_all) * 0.5), 5))
df_sorted = df_all.sort_values('RMSE')

# Color standard vs extra patients
colors = ['steelblue' if p in PATIENTS_21 else 'coral' for p in df_sorted.Patient]
ax.bar(range(len(df_sorted)), df_sorted.RMSE.values, color=colors, edgecolor='white')
ax.set_xticks(range(len(df_sorted)))
ax.set_xticklabels(df_sorted.Patient.values, rotation=45, ha='right')
ax.set_xlabel('Patient')
ax.set_ylabel('RMSE')
ax.set_title(f'GWO Search 2 — Per-Patient RMSE (H=10, {len(df_all)} Patients)', fontweight='bold')
ax.axhline(df_all.RMSE.mean(), color='red', ls='--', lw=1.5, label=f'Mean={df_all.RMSE.mean():.4f}')

# Legend for colors
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='Standard 21'),
    Patch(facecolor='coral', label='Extra patients'),
    plt.Line2D([0], [0], color='red', ls='--', lw=1.5, label=f'Mean={df_all.RMSE.mean():.4f}'),
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/rmse_bar_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7B.2 Per-Patient R² Bar — ALL patients ────────────────────────────────
fig, ax = plt.subplots(figsize=(max(16, len(df_all) * 0.5), 5))
df_sorted = df_all.sort_values('R2', ascending=False)

colors = ['steelblue' if p in PATIENTS_21 else 'coral' for p in df_sorted.Patient]
ax.bar(range(len(df_sorted)), df_sorted.R2.values, color=colors, edgecolor='white')
ax.set_xticks(range(len(df_sorted)))
ax.set_xticklabels(df_sorted.Patient.values, rotation=45, ha='right')
ax.set_xlabel('Patient')
ax.set_ylabel('R²')
ax.set_title(f'GWO Search 2 — Per-Patient R² (H=10, {len(df_all)} Patients)', fontweight='bold')
ax.axhline(df_all.R2.mean(), color='red', ls='--', lw=1.5, label=f'Mean={df_all.R2.mean():.4f}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/r2_bar_all.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7B.3 Metrics Box Plots ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, ['RMSE', 'MAE', 'R2']):
    # Split into standard 21 and extra
    data_21 = df_all[df_all.Patient.isin(PATIENTS_21)][metric]
    data_extra = df_all[~df_all.Patient.isin(PATIENTS_21)][metric]

    box_data = [data_21]
    labels = ['Standard 21']
    if len(data_extra) > 0:
        box_data.append(data_extra)
        labels.append('Extra')
    box_data.append(df_all[metric])
    labels.append('All')

    bp = ax.boxplot(box_data, labels=labels, patch_artist=True)
    colors_bp = ['steelblue', 'coral', 'mediumpurple'][:len(box_data)]
    for patch, color in zip(bp['boxes'], colors_bp):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(metric, fontweight='bold')

fig.suptitle(f'Metric Distributions — GWO Search 2 ({len(df_all)} Patients, H=10)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOT_DIR}/boxplots_all.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 — Summary

In [ ]:
print('=' * 80)
print('GWO SEARCH 2 — FINAL EVALUATION COMPLETE')
print('=' * 80)
print(f'\nOptimizer   : Grey Wolf Optimizer (mealpy)')
print(f'Pilot set   : [101, 103, 108, 119, 124] (harder — includes patient 101)')
print(f'Search      : 8 wolves × 4 epochs')
print(f'Pilot RMSE  : 0.030622')
print(f'\nBest Configuration:')
print(f'  n_filters   = {NUM_FILTERS}')
print(f'  dropout     = {DROPOUT_RATE}')
print(f'  lr          = {LEARNING_RATE}')
print(f'  n_blocks    = {NUM_BLOCKS} (fixed, RF=29)')
print(f'  kernel_size = {KERNEL_SIZE} (fixed)')
print(f'  batch_size  = {BATCH_SIZE} (fixed)')
print(f'\n── Part A: Standard 21 Patients (H=10) ──')
print(f'  Mean RMSE : {df_21.RMSE.mean():.6f}')
print(f'  Mean MAE  : {df_21.MAE.mean():.6f}')
print(f'  Mean R²   : {df_21.R2.mean():.4f}')
print(f'  CSV       : {CSV_21}')
print(f'\n── Part B: All {df_all.Patient.nunique()} Patients (H=10) ──')
print(f'  Mean RMSE : {df_all.RMSE.mean():.6f}')
print(f'  Mean MAE  : {df_all.MAE.mean():.6f}')
print(f'  Mean R²   : {df_all.R2.mean():.4f}')
print(f'  CSV       : {CSV_ALL}')
print(f'  Plots     : {PLOT_DIR}/')
print('=' * 80)